# omnicontext_gen_baseline
Generates the OmniContext benchmark under the unmodified model at the same engine settings as every domain arm: 512x512, `align_res=False`, references downscaled to `max_side=512`, seed 0, 50 steps, text 5.0 / image 2.0, `max_sequence_length=1024`, shipped negative prompt.

Reads `omnicontext/manifest.jsonl` and `omnicontext/images/` written by `build_omnicontext`. Row order is the frozen manifest order; outputs are indexed `{i:05d}.png` so the scorer joins on `i` exactly as the domain scorers do.

Companion to `omnicontext_gen_baseline_shipped`, which runs the same 400 rows at the released 1024x1024 `align_res=True` defaults. The pair separates engine settings from adapter effects: this notebook is the reference every OmniContext arm is measured against, the other is the calibration against the published number.

## 1. Clone at the pinned commit, install

In [1]:
import os, sys, subprocess
REPO   = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"   # the tree every arm of this thesis ran on
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
    # requirements.txt pins torch==2.6.0 and transformers==4.51.3, the stack every domain arm ran on.
    # An ad-hoc package list leaves Colab's current torch in place, and the arms then differ in the
    # whole software stack rather than in the adapter alone.
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"], capture_output=True, text=True).stdout.strip()
assert head == COMMIT, f"repo is at {head[:7]}, expected {COMMIT[:7]}. Run: git -C {REPO} checkout {COMMIT}"
sys.path.insert(0, REPO)
os.chdir(REPO)

import torch, transformers
assert transformers.__version__ == "4.51.3", transformers.__version__
assert torch.__version__.startswith("2.6.0"), f"torch {torch.__version__}, expected 2.6.0 from requirements.txt"
print(f"repo {head[:7]} | transformers {transformers.__version__} | torch {torch.__version__}")

installed.
repo 18e6f9d | transformers 4.51.3 | torch 2.6.0+cu124


## 2. Drive, references, manifest
References are tarred once and restored locally; 790 small reads from Drive per session otherwise dominate the run.

In [2]:
from google.colab import drive
import json
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
OC    = f"{DRIVE}/omnicontext"
TAR   = f"{OC}/images.tar"
LOCAL = "/content/oc"
IMGS  = f"{LOCAL}/images"

RUN     = "baseline"
DATASET = "omnicontext"
GEN     = f"{DRIVE}/gen/{DATASET}/{RUN}"
os.makedirs(GEN, exist_ok=True)

rows = [json.loads(l) for l in open(f"{OC}/manifest.jsonl", encoding="utf-8")]
REVISION = open(f"{OC}/revision.txt").read().strip()
EXPECTED = sum(len(r["input_images"]) for r in rows)

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

LOCALTAR = "/content/oc_images.tar"
if os.path.exists(TAR):
    subprocess.run(["cp", TAR, LOCALTAR], check=True)
else:
    # tar exits non-zero writing into the Drive FUSE mount, so build the archive on local
    # disk and copy the single file up. List form throughout: the Drive path has an
    # apostrophe and no shell is involved.
    r = subprocess.run(["tar", "-cf", LOCALTAR, "-C", OC, "images"], capture_output=True, text=True)
    assert os.path.exists(LOCALTAR), r.stderr[-800:]
    subprocess.run(["cp", LOCALTAR, TAR], check=True)
if n_files(IMGS) < EXPECTED:
    os.makedirs(LOCAL, exist_ok=True)
    subprocess.run(["tar", "-xf", LOCALTAR, "-C", LOCAL], check=True)

assert n_files(IMGS) == EXPECTED, (n_files(IMGS), EXPECTED)
missing = [p for r in rows for p in r["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, missing[:5]
print(f"rows {len(rows)} | references {EXPECTED} restored | dataset revision {REVISION}")
print(f"output -> {GEN}")

Mounted at /content/drive
rows 400 | references 790 restored | dataset revision 0f6aabbe8d3448d51184093372e980205d9980d3
output -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/omnicontext/baseline


## 3. Pipeline

In [3]:
import torch
from accelerate import Accelerator
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel

MODEL = "OmniGen2/OmniGen2"
accelerator = Accelerator(mixed_precision="bf16")
dtype = torch.bfloat16

pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
pipe.transformer = OmniGen2Transformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype)
pipe = pipe.to(accelerator.device)
print("pipeline ready on", accelerator.device, "| no adapter, unmodified weights")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Expected types for transformer: (<class 'omnigen2.models.transformers.transformer_omnigen2.OmniGen2Transformer2DModel'>,), got <class 'diffusers_modules.local.transformer_omnigen2.OmniGen2Transformer2DModel'>.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

pipeline ready on cuda | no adapter, unmodified weights


## 4. Engine
Identical to the domain arms. The two deviations from the released OmniContext defaults are the output size and `align_res`; both are applied to every arm alike and are what makes this leg comparable to the domain results rather than to the published leaderboard.

In [4]:
from PIL import Image, ImageOps

# shipped default from inference.py and omnicontext/inference.py; quality filter, not a domain negative
NEG = ("(((deformed))), blurry, over saturation, bad anatomy, disfigured, poorly drawn face, mutation, mutated, "
       "(extra_limb), (ugly), (poorly drawn hands), fused fingers, messy drawing, broken legs censor, censored, censor_bar")

def load_inputs(paths, max_side=512):   # refs downscaled; output size fixed at 512 via align_res=False
    out = []
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p).convert("RGB"))
        w, h = im.size
        s = max_side / max(w, h)
        if s < 1: w, h = int(w*s), int(h*s)
        w, h = max(16, (w//16)*16), max(16, (h//16)*16)
        out.append(im.resize((w, h), Image.LANCZOS))
    return out

def generate(instruction, input_images=None, negative=NEG, seed=0, steps=50, text_g=5.0, image_g=2.0, size=512):
    gen = torch.Generator(device=accelerator.device).manual_seed(seed)
    res = pipe(prompt=instruction, input_images=input_images,
               width=size, height=size, num_inference_steps=steps, max_sequence_length=1024,
               text_guidance_scale=text_g, image_guidance_scale=image_g, cfg_range=(0.0, 1.0),
               negative_prompt=negative, num_images_per_prompt=1, generator=gen, output_type="pil",
               align_res=False)
    return res.images[0]

print(f"run '{RUN}' on {DATASET}")
print("  instruction : unchanged")
print(f"  negative    : {'shipped default' if NEG else 'custom'}")
print("  sampler     : seed 0, 50 steps, text 5.0, image 2.0, cfg_range (0,1), max_sequence_length 1024")
print("  geometry    : 512x512, align_res=False, refs max_side=512")
print("  adapter     : none")

run 'baseline' on omnicontext
  instruction : unchanged
  negative    : shipped default
  sampler     : seed 0, 50 steps, text 5.0, image 2.0, cfg_range (0,1), max_sequence_length 1024
  geometry    : 512x512, align_res=False, refs max_side=512
  adapter     : none


## 5. Generate
Resumable: 400 rows is roughly 2.6 h at the domain arms' 23.5 s/img anchor.

In [5]:
import time
done, t0 = 0, time.time()
for i, r in enumerate(rows):
    out = f"{GEN}/{i:05d}.png"
    if os.path.exists(out):
        continue
    refs = load_inputs([os.path.join(IMGS, p) for p in r["input_images"]])
    img = generate(r["instruction"], input_images=refs, negative=NEG, seed=0)
    assert img.size == (512, 512), (i, img.size)
    img.save(out)
    done += 1
    if done % 10 == 0:
        print(f"generated {done} (at row {i+1}/{len(rows)}), {(time.time()-t0)/done:.1f}s/img")

total = len([f for f in os.listdir(GEN) if f.endswith(".png")])
print(f"\n{DATASET}/{RUN}: {total}/{len(rows)} images in {GEN}")
assert total == len(rows), "incomplete"

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

generated 400 (at row 400/400), 22.9s/img

omnicontext/baseline: 400/400 images in /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/omnicontext/baseline


## 6. Manifest

In [6]:
import datetime
with open(f"{GEN}/manifest.json", "w") as f:
    json.dump({
        "dataset": DATASET, "run": RUN, "rows": len(rows),
        "dataset_revision": REVISION, "repo_commit": head,
        "transformers": transformers.__version__,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "geometry": {"width": 512, "height": 512, "align_res": False, "ref_max_side": 512},
        "sampler": {"seed": 0, "steps": 50, "text_guidance_scale": 5.0,
                    "image_guidance_scale": 2.0, "cfg_range": [0.0, 1.0],
                    "max_sequence_length": 1024},
        "negative_prompt": NEG, "adapter": None,
        "matches_domain_arms": True,
        "written": datetime.datetime.now().isoformat(timespec="seconds"),
    }, f, indent=2)
print("manifest written")

manifest written
